# SoilGrids catalog explorer

Before fetching, it helps to know **what** SoilGrids publishes: which
properties, at which depths, and in what units. The `soilgrids` catalog is
config-as-code — a pydantic `Catalog` you can query offline (no network) to
build a request and to read the scaled-integer unit metadata.

This notebook lists the curated properties, renders their unit / depth /
quantile metadata as tables, and shows the did-you-mean error you get for a
typo.

## Load the catalog

`Catalog()` loads the bundled property catalog from disk. `parameters()` returns
the curated property ids.

In [1]:
import pandas as pd

from earthlens.soilgrids import Catalog

catalog = Catalog()
catalog.parameters()

['bdod',
 'cec',
 'cfvo',
 'clay',
 'nitrogen',
 'ocd',
 'ocs',
 'phh2o',
 'sand',
 'silt',
 'soc']

## Properties, units and scale factors

Each property row records its WCS endpoint, published depths and quantiles, and
the scaled-integer unit metadata (`unit`, `mapped_units`, `scale_factor`). The
table below is what you need to convert a downloaded raster to a physical unit:
divide the stored value by `scale_factor` to turn `mapped_units` into `unit`.

In [2]:
rows = [
    {
        "property": p,
        "title": catalog.get(p).title,
        "stored (mapped_units)": catalog.get(p).mapped_units,
        "scale_factor": catalog.get(p).scale_factor,
        "unit": catalog.get(p).unit,
    }
    for p in catalog.parameters()
]
pd.DataFrame(rows).set_index("property")

,title,stored (mapped_units),scale_factor,unit
property,,,,
bdod,Bulk density of the fine earth fraction,cg/cm3,100.0,kg/dm3
cec,Cation exchange capacity of the soil at pH 7,mmol(c)/kg,10.0,cmol(c)/kg
cfvo,Volumetric fraction of coarse fragments (> 2 mm),cm3/dm3,10.0,cm3/100cm3 (vol%)
clay,Proportion of clay particles (< 0.002 mm) in t...,g/kg,10.0,%
nitrogen,Total nitrogen (N),cg/kg,100.0,g/kg
ocd,Organic carbon density,hg/m3,10.0,kg/m3
ocs,Organic carbon stocks (0-30 cm),t/ha,10.0,kg/m2
phh2o,Soil pH in H2O,pH*10,10.0,pH
sand,Proportion of sand particles (> 0.05 mm) in th...,g/kg,10.0,%


## Depths and quantiles

Ten of the eleven properties publish the six **standard depth intervals**; the
`ocs` (organic carbon stock) property is the exception — a single `0-30cm`
interval. Every property offers the same five statistical **layers**.

In [3]:
depth_rows = [
    {
        "property": p,
        "n_depths": len(catalog.get(p).depths),
        "depths": ", ".join(catalog.get(p).depths),
    }
    for p in catalog.parameters()
]
pd.DataFrame(depth_rows).set_index("property")

,n_depths,depths
property,,
bdod,6,"0-5cm, 5-15cm, 15-30cm, 30-60cm, 60-100cm, 100..."
cec,6,"0-5cm, 5-15cm, 15-30cm, 30-60cm, 60-100cm, 100..."
cfvo,6,"0-5cm, 5-15cm, 15-30cm, 30-60cm, 60-100cm, 100..."
clay,6,"0-5cm, 5-15cm, 15-30cm, 30-60cm, 60-100cm, 100..."
nitrogen,6,"0-5cm, 5-15cm, 15-30cm, 30-60cm, 60-100cm, 100..."
ocd,6,"0-5cm, 5-15cm, 15-30cm, 30-60cm, 60-100cm, 100..."
ocs,1,0-30cm
phh2o,6,"0-5cm, 5-15cm, 15-30cm, 30-60cm, 60-100cm, 100..."
sand,6,"0-5cm, 5-15cm, 15-30cm, 30-60cm, 60-100cm, 100..."


In [4]:
# The quantile / layer tokens are the same for every property.
catalog.get("phh2o").quantiles

['Q0.05', 'Q0.5', 'Q0.95', 'mean', 'uncertainty']

## Did-you-mean on a typo

`get()` raises a `ValueError` with a suggestion when a property id is not
curated, so a typo fails fast with a hint rather than a silent miss.

In [5]:
try:
    catalog.get("clayy")
except ValueError as err:
    print(err)

'clayy' is not in the SoilGrids property catalog. Known properties: ['bdod', 'cec', 'cfvo', 'clay', 'nitrogen', 'ocd', 'ocs', 'phh2o', 'sand', 'silt', 'soc']. Did you mean 'clay'?


## Takeaway

- `Catalog().parameters()` lists the 11 curated properties; `Catalog().get(id)`
  returns a row with its depths, quantiles, and unit / scale metadata.
- Divide a downloaded raster by `scale_factor` to convert to the physical
  `unit`.
- Feed the property / depth / quantile ids straight into
  `EarthLens(data_source="soilgrids", variables=, depths=, quantiles=)`.